# YouTube · Instagram · X 공통 컬럼 병합 및 값 정규화

세 플랫폼의 정규화 테이블을 먼저 관계형 구조에 맞게 복원한 뒤, 모든 플랫폼에서 의미가 공통인 컬럼만 세로 결합한다. 최종 컬럼은 `platform`, `post_id`, `post_url`, `created_at`, `content`, `like_count`, `comment_count`이다.

정규화 규칙: ID는 문자열로 보존하고, URL은 `https`, 소문자 호스트, 불필요한 끝 슬래시 제거 형식으로 맞춘다. 유튜브의 `?v=`처럼 자원 식별에 필요한 쿼리는 보존하고 프래그먼트는 제거한다. 원본 `created_at`의 기입값 자체를 한국 표준시(KST)로 간주하므로 시각을 더하거나 빼지 않고 `+09:00` 정보만 명시한다. 텍스트는 Unicode NFKC와 연속 공백 정리를 적용한다. 반응 수치는 nullable 정수로 변환하며, `(platform, post_id)`를 복합키로 중복을 검증한다.

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
import re
import unicodedata

import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / '02_csv').is_dir() and (candidate / '03_code').is_dir():
            return candidate
    raise FileNotFoundError('02_csv와 03_code가 있는 프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
YOUTUBE_DIR = PROJECT_ROOT / '02_csv' / '01_youtube' / '1. videos' / '02. 정제' / '02. 3차_정규화_&_컬럼명_통합'
INSTAGRAM_DIR = PROJECT_ROOT / '02_csv' / '02_instagram'
X_DIR = PROJECT_ROOT / '02_csv' / '03_x' / 'normalized'
OUTPUT_DIR = PROJECT_ROOT / '02_csv' / '06_merge'
OUTPUT_PATH = OUTPUT_DIR / 'social_platforms_merged_normalized.csv'

YOUTUBE_DIR, INSTAGRAM_DIR, X_DIR, OUTPUT_PATH

## 1. 대상 파일 검색 및 로드

유튜브는 지정 폴더의 모든 CSV, 인스타그램은 하위 폴더까지 파일명에 `3nf`가 있는 모든 CSV, X는 `normalized` 폴더의 모든 CSV를 사용한다. 인스타그램 `temp` 파일은 루트 파일과 같은 데이터를 이전 컬럼명으로 보관한 복제본이므로, 컬럼명을 통일한 후 업무 키로 중복을 제거해 이중 집계를 방지한다.

In [ ]:
def read_csv_as_string(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype='string', encoding='utf-8-sig', keep_default_na=True)


youtube_files = sorted(YOUTUBE_DIR.glob('*.csv'))
instagram_files = sorted(p for p in INSTAGRAM_DIR.rglob('*.csv') if '3nf' in p.name.lower())
x_files = sorted(X_DIR.glob('*.csv'))

if not youtube_files or not instagram_files or not x_files:
    raise FileNotFoundError('한 플랫폼 이상의 대상 CSV가 없습니다.')

youtube_tables = {p.name: read_csv_as_string(p) for p in youtube_files}
instagram_tables = {str(p.relative_to(INSTAGRAM_DIR)): read_csv_as_string(p) for p in instagram_files}
x_tables = {p.name: read_csv_as_string(p) for p in x_files}

file_inventory = pd.DataFrame(
    [("youtube", str(p.relative_to(PROJECT_ROOT)), len(youtube_tables[p.name])) for p in youtube_files]
    + [("instagram", str(p.relative_to(PROJECT_ROOT)), len(instagram_tables[str(p.relative_to(INSTAGRAM_DIR))])) for p in instagram_files]
    + [("x", str(p.relative_to(PROJECT_ROOT)), len(x_tables[p.name])) for p in x_files],
    columns=['platform', 'source_file', 'row_count'],
)
print(file_inventory.to_string(index=False))

## 2. 플랫폼별 관계 복원

최종 공통 컬럼에 포함되지 않는 차원/브리지 테이블도 빠뜨리지 않는다. 유튜브 채널, 인스타그램 미디어 유형·키워드, X 원본 파일·키워드 테이블을 각각 조인하고 외래키를 검증한 다음 공통 컬럼을 선택한다.

In [ ]:
# YouTube: video -> channel (many-to-one)
youtube_channels = youtube_tables['youtube_analysis_channel_table.csv'].drop_duplicates('account_id')
youtube_videos = youtube_tables['youtube_analysis_video_table.csv'].copy()
unknown_youtube_accounts = set(youtube_videos['account_id'].dropna()) - set(youtube_channels['account_id'].dropna())
if unknown_youtube_accounts:
    raise ValueError(f'YouTube 채널 외래키 누락: {len(unknown_youtube_accounts):,}개')
youtube_full = youtube_videos.merge(youtube_channels, on='account_id', how='left', validate='many_to_one')
youtube_full['post_url'] = 'https://www.youtube.com/watch?v=' + youtube_full['post_id'].astype('string')
youtube_full['content'] = (
    youtube_full['post_title'].fillna('').str.strip()
    + ' '
    + youtube_full['description'].fillna('').str.strip()
).str.strip()

# Instagram: root/temp의 구 컬럼명을 현재 컬럼명으로 통일한 뒤 중복 제거
instagram_aliases = {
    'id': 'post_id', 'caption': 'description', 'permalink': 'post_url',
    'timestamp': 'created_at', 'comments_count': 'comment_count',
}
instagram_posts_parts = []
instagram_media_parts = []
instagram_keyword_parts = []
instagram_bridge_parts = []
for relative_name, frame in instagram_tables.items():
    normalized = frame.rename(columns=instagram_aliases).copy()
    if relative_name.endswith('3nf_posts.csv'):
        instagram_posts_parts.append(normalized)
    elif relative_name.endswith('3nf_media_types.csv'):
        instagram_media_parts.append(normalized)
    elif relative_name.endswith('3nf_post_keywords.csv'):
        instagram_bridge_parts.append(normalized)
    elif relative_name.endswith('3nf_keywords.csv'):
        instagram_keyword_parts.append(normalized)

instagram_posts = pd.concat(instagram_posts_parts, ignore_index=True).drop_duplicates('post_id', keep='first')
instagram_media = pd.concat(instagram_media_parts, ignore_index=True).drop_duplicates('media_type_id')
instagram_keywords = pd.concat(instagram_keyword_parts, ignore_index=True).drop_duplicates('keyword_id')
instagram_bridge = pd.concat(instagram_bridge_parts, ignore_index=True).drop_duplicates(['post_id', 'keyword_id'])

if not set(instagram_posts['media_type_id'].dropna()).issubset(set(instagram_media['media_type_id'].dropna())):
    raise ValueError('Instagram media_type_id 외래키가 media_types에 없습니다.')
if not set(instagram_bridge['post_id'].dropna()).issubset(set(instagram_posts['post_id'].dropna())):
    raise ValueError('Instagram 키워드 브리지의 post_id가 posts에 없습니다.')
if not set(instagram_bridge['keyword_id'].dropna()).issubset(set(instagram_keywords['keyword_id'].dropna())):
    raise ValueError('Instagram 키워드 브리지의 keyword_id가 keywords에 없습니다.')

instagram_keyword_text = (
    instagram_bridge.merge(instagram_keywords, on='keyword_id', how='left', validate='many_to_one')
    .groupby('post_id', sort=False)['keyword']
    .agg(lambda values: ', '.join(dict.fromkeys(v for v in values.dropna())))
    .rename('keywords')
)
instagram_full = (
    instagram_posts.merge(instagram_media, on='media_type_id', how='left', validate='many_to_one')
    .merge(instagram_keyword_text, on='post_id', how='left', validate='one_to_one')
)
instagram_full['content'] = instagram_full['description']

# X: post -> source file, post -> keyword
x_posts = x_tables['x_posts.csv'].copy()
x_sources = x_tables['x_post_sources.csv'].merge(
    x_tables['x_source_files.csv'], on='source_file', how='left', validate='many_to_one'
)
x_keyword_text = (
    x_tables['x_post_keywords.csv']
    .sort_values(['post_id', 'keyword_order'], kind='stable')
    .groupby('post_id', sort=False)['keyword']
    .agg(lambda values: ', '.join(dict.fromkeys(v for v in values.dropna())))
    .rename('keywords')
)
if not set(x_sources['post_id'].dropna()).issubset(set(x_posts['post_id'].dropna())):
    raise ValueError('X 출처 테이블의 post_id가 posts에 없습니다.')
if not set(x_tables['x_post_keywords.csv']['keyword'].dropna()).issubset(set(x_tables['x_keywords.csv']['keyword'].dropna())):
    raise ValueError('X 게시물 키워드가 keywords 차원에 없습니다.')
x_source_text = (
    x_sources.groupby('post_id', sort=False)['source_file']
    .agg(lambda values: ', '.join(dict.fromkeys(v for v in values.dropna())))
    .rename('source_files')
)
x_full = (
    x_posts.merge(x_keyword_text, on='post_id', how='left', validate='one_to_one')
    .merge(x_source_text, on='post_id', how='left', validate='one_to_one')
)
x_full['content'] = x_full['text']

print({'youtube': len(youtube_full), 'instagram': len(instagram_full), 'x': len(x_full)})

## 3. 공통 스키마 및 값 정규화

`post_url`은 유튜브 ID로 생성할 수 있어 세 플랫폼 공통 속성으로 포함했다. 유튜브의 제목과 설명은 한 행의 `content`로 합치고, 인스타그램 설명과 X 본문도 같은 컬럼으로 매핑한다.

In [ ]:
COMMON_COLUMNS = [
    'platform', 'post_id', 'post_url', 'created_at',
    'content', 'like_count', 'comment_count',
]


def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    normalized = unicodedata.normalize('NFKC', str(value))
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized if normalized else pd.NA


def normalize_url(value):
    if pd.isna(value):
        return pd.NA
    raw = str(value).strip()
    if not raw:
        return pd.NA
    parts = urlsplit(raw if '://' in raw else 'https://' + raw)
    path = parts.path.rstrip('/') or '/'
    return urlunsplit(('https', parts.netloc.lower(), path, parts.query, ''))


def normalize_count(series: pd.Series, name: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    invalid = series.notna() & numeric.isna()
    if invalid.any():
        raise ValueError(f'{name}: 숫자로 변환할 수 없는 값 {int(invalid.sum()):,}개')
    non_integer = numeric.notna() & numeric.mod(1).ne(0)
    if non_integer.any():
        raise ValueError(f'{name}: 정수가 아닌 값 {int(non_integer.sum()):,}개')
    if numeric.lt(0).fillna(False).any():
        raise ValueError(f'{name}: 음수 값이 있습니다.')
    return numeric.astype('Int64')


def to_common(frame: pd.DataFrame, platform: str) -> pd.DataFrame:
    result = frame[['post_id', 'post_url', 'created_at', 'content', 'like_count', 'comment_count']].copy()
    result.insert(0, 'platform', platform)
    result['post_id'] = result['post_id'].astype('string').str.strip()
    if result['post_id'].isna().any() or result['post_id'].eq('').any():
        raise ValueError(f'{platform}: 빈 post_id가 있습니다.')
    result['post_url'] = result['post_url'].map(normalize_url).astype('string')
    # 원본에 기입된 시·분·초 자체가 KST이다. 기존 Z/+HH:MM 표시는 제거하고
    # 시각 값은 이동하지 않은 채 Asia/Seoul 시간대만 부여한다.
    kst_wall_clock = (
        result['created_at'].astype('string').str.strip()
        .str.replace(r'(?:Z|[+-]\d{2}:?\d{2})$', '', regex=True)
        .str.strip()
    )
    timestamps = pd.to_datetime(kst_wall_clock, errors='coerce')
    if timestamps.isna().any():
        raise ValueError(f'{platform}: 변환하지 못한 created_at {int(timestamps.isna().sum()):,}개')
    result['created_at'] = (
        timestamps.dt.tz_localize('Asia/Seoul')
        .dt.strftime('%Y-%m-%dT%H:%M:%S+09:00')
        .astype('string')
    )
    result['content'] = result['content'].map(normalize_text).astype('string')
    result['like_count'] = normalize_count(result['like_count'], f'{platform}.like_count')
    result['comment_count'] = normalize_count(result['comment_count'], f'{platform}.comment_count')
    return result[COMMON_COLUMNS]


youtube_common = to_common(youtube_full, 'youtube')
instagram_common = to_common(instagram_full, 'instagram')
x_common = to_common(x_full, 'x')

merged = pd.concat([youtube_common, instagram_common, x_common], ignore_index=True)
duplicate_key = merged.duplicated(['platform', 'post_id'], keep=False)
if duplicate_key.any():
    conflicts = merged.loc[duplicate_key].sort_values(['platform', 'post_id'])
    if conflicts.groupby(['platform', 'post_id'], dropna=False)[COMMON_COLUMNS].nunique(dropna=False).gt(1).any(axis=None):
        raise ValueError('동일 (platform, post_id)에 서로 다른 값이 있습니다.')
    merged = merged.drop_duplicates(['platform', 'post_id'], keep='first')

platform_order = pd.Categorical(merged['platform'], categories=['youtube', 'instagram', 'x'], ordered=True)
merged = (
    merged.assign(_platform_order=platform_order)
    .sort_values(['_platform_order', 'created_at', 'post_id'], kind='stable')
    .drop(columns='_platform_order')
    .reset_index(drop=True)
)
merged.head()

## 4. 저장 및 검증

CSV는 Excel에서도 한글이 깨지지 않도록 UTF-8 BOM(`utf-8-sig`)으로 저장한다. 저장 후 다시 읽어 행 수·컬럼·복합키·자료형을 재검증한다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', lineterminator='\n')

reloaded = pd.read_csv(OUTPUT_PATH, dtype={'platform': 'string', 'post_id': 'string'}, encoding='utf-8-sig')
expected_rows = len(youtube_common) + len(instagram_common) + len(x_common)
youtube_saved = reloaded.loc[reloaded['platform'].eq('youtube')]
x_saved = reloaded.loc[reloaded['platform'].eq('x')]
checks = {
    'columns_match': reloaded.columns.tolist() == COMMON_COLUMNS,
    'row_count_match': len(reloaded) == expected_rows == len(merged),
    'composite_key_unique': not reloaded.duplicated(['platform', 'post_id']).any(),
    'platforms_complete': set(reloaded['platform']) == {'youtube', 'instagram', 'x'},
    'post_url_complete': reloaded['post_url'].notna().all(),
    'youtube_url_keeps_video_id': youtube_saved['post_url'].str.split('v=', n=1).str[-1].eq(youtube_saved['post_id']).all(),
    'x_url_keeps_post_id': x_saved['post_url'].str.rsplit('/', n=1).str[-1].eq(x_saved['post_id']).all(),
    'created_at_kst_iso': reloaded['created_at'].str.fullmatch(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\+09:00').all(),
    'counts_nonnegative': (
        reloaded[['like_count', 'comment_count']].isna()
        | reloaded[['like_count', 'comment_count']].ge(0)
    ).all().all(),
}
if not all(checks.values()):
    raise AssertionError({k: v for k, v in checks.items() if not v})

summary = (
    reloaded.groupby('platform', sort=False)
    .agg(
        rows=('post_id', 'size'),
        unique_posts=('post_id', 'nunique'),
        min_created_at=('created_at', 'min'),
        max_created_at=('created_at', 'max'),
        missing_content=('content', lambda s: int(s.isna().sum())),
        missing_like_count=('like_count', lambda s: int(s.isna().sum())),
        missing_comment_count=('comment_count', lambda s: int(s.isna().sum())),
    )
)
print(summary.to_string())
print(f'\n총 {len(reloaded):,}행 저장: {OUTPUT_PATH}')
print('검증:', checks)

## 정규화 설명

- **스키마 정규화**: 플랫폼마다 이름이 다른 본문·시간·댓글 컬럼을 하나의 공통 명칭으로 매핑했다. 세 플랫폼에 없는 속성은 최종 테이블에서 제외했다.
- **키 정규화**: 큰 숫자형 ID가 지수 표기나 반올림으로 손상되지 않도록 문자열로 읽고 보존했다. `(platform, post_id)`를 최종 복합키로 사용한다.
- **시간 정규화**: 원본에 기입된 시·분·초 자체를 한국 표준시로 간주한다. UTC 변환이나 9시간 가감 없이 기존 시각을 그대로 유지하고 `YYYY-MM-DDTHH:MM:SS+09:00` 형식으로 시간대 정보만 명시한다.
- **문자 정규화**: 호환 문자와 전각/반각 차이를 줄이기 위해 NFKC를 적용하고, 줄바꿈·탭·연속 공백을 한 칸으로 통일했다.
- **수치 정규화**: 좋아요·댓글 수를 nullable 정수로 변환하고 음수 및 비정수 값을 검증했다.
- **관계 복원 및 중복 방지**: 각 플랫폼의 차원/브리지 테이블을 먼저 조인했다. 인스타그램 `temp` 복제본은 업무 키로 중복 제거하여 행이 두 번 집계되지 않게 했다.